In [2]:
# Cell 1 — Imports & GPU
import os, glob, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, TopKPooling, global_mean_pool
from skimage.segmentation import slic
from skimage.color import rgb2lab
from skimage import graph as sk_graph

os.environ["CUDA_VISIBLE_DEVICES"] = "6"   # ← change to your GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))

Image.MAX_IMAGE_PIXELS = None

Device: cuda
GPU   : NVIDIA H200 MIG 3g.71gb


In [3]:
# Cell 3 — Graph construction functions (identical to LC25000)
def generate_superpixels(image_tensor):
    """
    image_tensor: [3, H, W] float32 in [0,1] (after ToTensor, no Normalize)
    Returns: segments array [H, W] with integer labels starting at 0
    Paper §4.1.2: SLIC in CIE-Lab color space, n_segments=300
    """
    img_np  = image_tensor.permute(1, 2, 0).cpu().numpy()  # [H, W, 3]
    img_lab = rgb2lab(img_np)                               # CIE-Lab (paper)
    segments = slic(
        img_lab,
        n_segments=300,
        compactness=10,
        sigma=1,
        start_label=0,
        channel_axis=-1
    )
    return segments


def create_node_features(image_tensor, segments):
    """
    Paper Eq.6: x_i = softmax([a_i/224, b_i/224, R_i/224, G_i/224, B_i/224])
    Returns: np.array [num_segments, 5]
    """
    img_np = image_tensor.permute(1, 2, 0).cpu().numpy()  # [H, W, 3]
    nodes  = []
    for seg_id in sorted(np.unique(segments)):
        mask   = (segments == seg_id)
        coords = np.argwhere(mask)
        y_c = coords[:, 0].mean() / 224.0
        x_c = coords[:, 1].mean() / 224.0
        color = img_np[mask].mean(axis=0)
        feat = np.array([
            x_c,
            y_c,
            color[0] / 224.0 * 255.0,
            color[1] / 224.0 * 255.0,
            color[2] / 224.0 * 255.0,
        ], dtype=np.float32)
        nodes.append(feat)
    nodes = np.array(nodes, dtype=np.float32)
    exp   = np.exp(nodes - nodes.max(axis=1, keepdims=True))
    nodes = exp / exp.sum(axis=1, keepdims=True)
    return nodes


def create_edges(segments):
    """
    Paper Eq.5: A_ij=1 if superpixels i and j are adjacent.
    """
    unique = sorted(np.unique(segments))
    remap  = np.zeros(segments.max() + 1, dtype=np.int32)
    for new, old in enumerate(unique):
        remap[old] = new
    seg_r = remap[segments]

    dummy = np.zeros((*seg_r.shape, 3), dtype=np.float32)
    rag   = sk_graph.rag_mean_color(dummy, seg_r)

    edges = []
    for u, v in rag.edges:
        if u < len(unique) and v < len(unique):
            edges += [[u, v], [v, u]]
    if not edges:
        return torch.zeros((2, 0), dtype=torch.long)
    return torch.tensor(edges, dtype=torch.long).t().contiguous()


def image_to_graph(image_tensor, label):
    """Full pipeline: image → PyG Data object"""
    segments   = generate_superpixels(image_tensor)
    node_feats = create_node_features(image_tensor, segments)
    edge_index = create_edges(segments)
    return Data(
        x          = torch.tensor(node_feats, dtype=torch.float),
        edge_index = edge_index,
        y          = torch.tensor([label], dtype=torch.long)
    )

print("Graph construction functions defined.")
print("Node feature dim: 5  (x/224, y/224, R/224, G/224, B/224 — paper Eq.6)")

Graph construction functions defined.
Node feature dim: 5  (x/224, y/224, R/224, G/224, B/224 — paper Eq.6)


In [4]:
# Cell 5 — BRACS Dataset
DATA_DIR = "BRACS_Dataset/BRACS_RoI/latest_version"  # ← change if needed

class HistologyDataset(Dataset):
    SPLITS   = ["train", "val", "test"]
    IMG_EXTS = (".png", ".jpg", ".jpeg", ".tif", ".bmp")

    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.images   = []
        self.labels   = []

        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor()
        ])

        class_names = set()
        for split in self.SPLITS:
            sp = os.path.join(root_dir, split)
            if not os.path.isdir(sp): continue
            for cls in os.listdir(sp):
                cls_path = os.path.join(sp, cls)
                if os.path.isdir(cls_path) and not cls.startswith('.'):
                    class_names.add(cls)

        self.class_to_idx = {cls: i for i, cls in enumerate(sorted(class_names))}

        # Collect all images — verify each file is readable before adding
        bad_files = []
        for split in self.SPLITS:
            sp = os.path.join(root_dir, split)
            if not os.path.isdir(sp): continue
            for cls in os.listdir(sp):
                cls_path = os.path.join(sp, cls)
                if not os.path.isdir(cls_path) or cls.startswith('.'): continue
                for file in os.listdir(cls_path):
                    if file.lower().endswith(self.IMG_EXTS):
                        fpath = os.path.join(cls_path, file)
                        # Skip files that can't be opened at all
                        if not os.path.isfile(fpath) or os.path.getsize(fpath) == 0:
                            bad_files.append(fpath)
                            continue
                        self.images.append(fpath)
                        self.labels.append(self.class_to_idx[cls])

        if bad_files:
            print(f"  ⚠ Skipped {len(bad_files)} missing/empty files during init")

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        try:
            from PIL import ImageFile
            ImageFile.LOAD_TRUNCATED_IMAGES = True
            img = Image.open(self.images[idx]).convert("RGB")
            img = self.transform(img)
            return image_to_graph(img, self.labels[idx])
        except Exception as e:
            print(f"  WARNING: Bad image idx={idx} path={self.images[idx]}  [{e}]")
            # Return a black image graph so the sample is skipped gracefully in precompute
            dummy = torch.zeros(3, 224, 224)
            return image_to_graph(dummy, self.labels[idx])


raw_dataset = HistologyDataset(DATA_DIR)
NUM_CLASSES = len(raw_dataset.class_to_idx)
print("Dataset size:", len(raw_dataset))
print("Classes     :", raw_dataset.class_to_idx)
assert NUM_CLASSES == 7, f"Expected 7 classes, got {NUM_CLASSES}. Check DATA_DIR."

Dataset size: 4168
Classes     : {'0_N': 0, '1_PB': 1, '2_UDH': 2, '3_FEA': 3, '4_ADH': 4, '5_DCIS': 5, '6_IC': 6}


In [5]:
# Cell 7 — Precompute and cache superpixel graphs
CACHE_DIR = "graph_cache_bracs-re"

BAD_INDICES = []   # track which indices produced errors

def precompute_graphs(dataset, cache_dir):
    os.makedirs(cache_dir, exist_ok=True)
    total   = len(dataset)
    t0      = time.time()
    skipped = 0
    for idx in range(total):
        cache_path = os.path.join(cache_dir, f"graph_{idx:06d}.pt")
        if os.path.exists(cache_path):
            skipped += 1
            continue
        try:
            data = dataset[idx]
            torch.save(data, cache_path)
        except OSError as e:
            print(f"  OSError  idx={idx:>5}  file={dataset.images[idx]}  [{e}]")
            BAD_INDICES.append(idx)
            continue
        except Exception as e:
            print(f"  Warning: skipping idx {idx} — {type(e).__name__}: {e}")
            BAD_INDICES.append(idx)
            continue
        if idx % 200 == 0:
            elapsed = time.time() - t0
            eta     = (elapsed / max(idx - skipped + 1, 1)) * (total - idx)
            print(f"  [{idx:>5}/{total}]  elapsed {elapsed/60:.1f}min  ETA {eta/60:.1f}min")
    print(f"Done. Total cached: {len(os.listdir(cache_dir))} files")
    if BAD_INDICES:
        print(f"  ⚠ {len(BAD_INDICES)} images failed — indices: {BAD_INDICES}")
        # Save bad index list so you can inspect/re-download them
        import json
        with open("bracs_bad_image_indices.json", "w") as f:
            json.dump({"bad_indices": BAD_INDICES,
                       "bad_paths": [dataset.images[i] for i in BAD_INDICES]}, f, indent=2)
        print("  Saved → bracs_bad_image_indices.json")

precompute_graphs(raw_dataset, CACHE_DIR)

Done. Total cached: 4168 files


In [6]:
# Cell 8 — Load cached graphs + verify labels
class CachedDataset(Dataset):
    def __init__(self, cache_dir):
        self.files = sorted(glob.glob(os.path.join(cache_dir, "graph_*.pt")))
        print(f"Loaded {len(self.files)} cached graphs from {cache_dir}")
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        return torch.load(self.files[idx], weights_only=False)

cached_dataset = CachedDataset(CACHE_DIR)

# Verify labels are clean 0–6
print("\nVerifying labels...")
sample_labels = []
for f in sorted(glob.glob(os.path.join(CACHE_DIR, "graph_*.pt")))[:500]:
    d = torch.load(f, weights_only=False)
    sample_labels.append(d.y.item())
print(f"  Unique labels (first 500): {sorted(set(sample_labels))}")
assert max(sample_labels) <= 6, "Labels out of range — re-run dataset creation"
assert min(sample_labels) >= 0, "Negative label found"
print("  Label check passed.")

loader = DataLoader(
    cached_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True
)
print(f"DataLoader ready: {len(loader)} batches")

Loaded 4168 cached graphs from graph_cache_bracs-re

Verifying labels...
  Unique labels (first 500): [0, 2]
  Label check passed.
DataLoader ready: 131 batches


In [7]:
# Cell 10 — GAT Model
# num_classes=7 for BRACS (was 5 for LC25000)
class GNNModel(nn.Module):
    """
    Paper §4.1.3 GNN feature extractor:
      - 3-layer GAT (heads: 1, 2, 4)
      - BatchNorm after each GAT layer
      - TopKPooling: 300 → 196 nodes (ratio=196/300)
      - Linear projection: 256 → 1024 (match CNN feature dim)
    Input node feature dim: 5 (paper Eq.6)
    """
    def __init__(self, in_features=5, num_classes=7, dropout=0.3):
        super().__init__()
        self.dropout = dropout

        # Layer 1: 1 head, concat=False → out: 32
        self.gat1 = GATConv(in_features, 32,  heads=1, concat=False)
        self.bn1  = nn.BatchNorm1d(32)

        # Layer 2: 2 heads, concat=True → out: 64×2=128
        self.gat2 = GATConv(32, 64, heads=2, concat=True)
        self.bn2  = nn.BatchNorm1d(128)

        # Layer 3: 4 heads, concat=False → out: 256
        self.gat3 = GATConv(128, 256, heads=4, concat=False)
        self.bn3  = nn.BatchNorm1d(256)

        # TopKPooling: 300 → 196 nodes (to match CNN spatial shape 14×14=196)
        self.topk_pool = TopKPooling(in_channels=256, ratio=196/300)

        # Project 256 → 1024 to match CNN feature dimension for AFF
        self.proj = nn.Linear(256, 1024)

        # Classification head (used during GNN pre-training only)
        self.fc = nn.Linear(1024, num_classes)

    def forward(self, x, edge_index, batch, return_node_features=False):
        # GAT layer 1
        x = F.relu(self.bn1(self.gat1(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        # GAT layer 2
        x = F.relu(self.bn2(self.gat2(x, edge_index)))
        x = F.dropout(x, p=self.dropout, training=self.training)

        # GAT layer 3
        x = F.relu(self.bn3(self.gat3(x, edge_index)))

        # TopKPooling: 300 → 196 nodes
        x, edge_index, _, batch, _, _ = self.topk_pool(x, edge_index, batch=batch)

        # Project to 1024
        x = self.proj(x)   # [total_kept_nodes, 1024]

        if return_node_features:
            return x, batch

        # Classification head
        x = global_mean_pool(x, batch)
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.fc(x)


model     = GNNModel(in_features=5, num_classes=NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=10, min_lr=1e-6
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {total_params:,}")
print(f"Input node features : 5 (paper Eq.6)")
print(f"Output classes      : {NUM_CLASSES} (BRACS 7-class)")

GNNModel(
  (gat1): GATConv(5, 32, heads=1)
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (gat2): GATConv(32, 64, heads=2)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (gat3): GATConv(128, 256, heads=4)
  (bn3): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (topk_pool): TopKPooling(256, ratio=0.6533333333333333, multiplier=1.0)
  (proj): Linear(in_features=256, out_features=1024, bias=True)
  (fc): Linear(in_features=1024, out_features=7, bias=True)
)

Trainable parameters: 409,543
Input node features : 5 (paper Eq.6)
Output classes      : 7 (BRACS 7-class)


In [8]:
# Cell 12 — Train GNN
EPOCHS          = 150
PATIENCE        = 20
best_acc        = 0.0
patience_ctr    = 0
best_model_path = "gnn_model_bracs_best-re.pth"

print(f"Training GNN for up to {EPOCHS} epochs (patience={PATIENCE})...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = correct = total = 0
    t0 = time.time()

    for i, data in enumerate(loader):
        data = data.to(device)
        optimizer.zero_grad()
        out  = model(data.x, data.edge_index, data.batch)
        loss = criterion(out, data.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        pred        = out.argmax(dim=1)
        correct    += (pred == data.y).sum().item()
        total      += data.y.size(0)

        if i % 20 == 0:   # more frequent logging — BRACS has fewer batches
            print(f"  Ep {epoch} | Batch {i:>4}/{len(loader)} | "
                  f"loss={loss.item():.4f} | acc={correct/max(total,1):.3f}")

    train_acc = correct / total
    avg_loss  = total_loss / len(loader)
    scheduler.step(train_acc)
    elapsed   = time.time() - t0
    print(f"Epoch {epoch:03d} — Loss={avg_loss:.4f} | Acc={train_acc:.4f} | {elapsed:.1f}s")

    if train_acc > best_acc:
        best_acc     = train_acc
        patience_ctr = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✓ Best acc={best_acc:.4f} — saved")
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest training accuracy: {best_acc:.4f}")
torch.save(model.state_dict(), "gnn_model_bracs_final-re.pth")

Training GNN for up to 150 epochs (patience=20)...
  Ep 1 | Batch    0/131 | loss=1.9494 | acc=0.125
  Ep 1 | Batch   20/131 | loss=1.9615 | acc=0.207
  Ep 1 | Batch   40/131 | loss=1.8867 | acc=0.206
  Ep 1 | Batch   60/131 | loss=1.8112 | acc=0.215
  Ep 1 | Batch   80/131 | loss=1.8886 | acc=0.222
  Ep 1 | Batch  100/131 | loss=1.6895 | acc=0.234
  Ep 1 | Batch  120/131 | loss=1.8292 | acc=0.241
Epoch 001 — Loss=1.8496 | Acc=0.2454 | 7.8s
  ✓ Best acc=0.2454 — saved
  Ep 2 | Batch    0/131 | loss=1.7439 | acc=0.344
  Ep 2 | Batch   20/131 | loss=1.7021 | acc=0.298
  Ep 2 | Batch   40/131 | loss=1.7636 | acc=0.303
  Ep 2 | Batch   60/131 | loss=1.7187 | acc=0.303
  Ep 2 | Batch   80/131 | loss=1.8762 | acc=0.301
  Ep 2 | Batch  100/131 | loss=1.8258 | acc=0.303
  Ep 2 | Batch  120/131 | loss=1.6994 | acc=0.301
Epoch 002 — Loss=1.7459 | Acc=0.2992 | 4.3s
  ✓ Best acc=0.2992 — saved
  Ep 3 | Batch    0/131 | loss=1.6411 | acc=0.219
  Ep 3 | Batch   20/131 | loss=1.5320 | acc=0.289
  Ep 

In [11]:
# Cell 14 — Extract GNN node features
MAX_NODES = 196     # matches CNN spatial shape 14×14
FEAT_DIM  = 1024    # matches CNN channel dim

model.load_state_dict(torch.load(best_model_path, weights_only=True))
model.eval()

all_node_features = []
all_labels        = []

print("Extracting GNN node features...")
with torch.no_grad():
    for batch_idx, data in enumerate(loader):
        data       = data.to(device)
        node_feats, batch_vec = model(
            data.x, data.edge_index, data.batch,
            return_node_features=True
        )
        labels_b   = data.y
        batch_size = labels_b.size(0)

        for img_idx in range(batch_size):
            mask      = (batch_vec == img_idx)
            img_nodes = node_feats[mask]          # [n_kept, 1024]
            n         = img_nodes.size(0)

            if n < MAX_NODES:
                pad       = torch.zeros(MAX_NODES - n, FEAT_DIM, device=device)
                img_nodes = torch.cat([img_nodes, pad], dim=0)
            else:
                img_nodes = img_nodes[:MAX_NODES]  # [196, 1024]

            all_node_features.append(img_nodes.cpu())
        all_labels.extend(labels_b.cpu().tolist())

        if batch_idx % 20 == 0:
            print(f"  Batch {batch_idx}/{len(loader)}")

gnn_features  = torch.stack(all_node_features)          # [3892, 196, 1024]
labels_tensor = torch.tensor(all_labels, dtype=torch.long)

print(f"\nGNN feature shape : {gnn_features.shape}")
print(f"Labels shape      : {labels_tensor.shape}")
print(f"Unique labels     : {torch.unique(labels_tensor).tolist()}")
assert gnn_features.shape[1:] == (196, 1024), "Shape mismatch!"
assert labels_tensor.min() >= 0 and labels_tensor.max() <= 6, \
    f"Label range error! Got [{labels_tensor.min()}, {labels_tensor.max()}]"
# Use actual cached count, not hardcoded 3892
print(f"  Total extracted: {gnn_features.shape[0]} (original dataset may differ due to skipped bad images)")
print("All assertions passed.")

Extracting GNN node features...
  Batch 0/131
  Batch 20/131
  Batch 40/131
  Batch 60/131
  Batch 80/131
  Batch 100/131
  Batch 120/131

GNN feature shape : torch.Size([4168, 196, 1024])
Labels shape      : torch.Size([4168])
Unique labels     : [0, 1, 2, 3, 4, 5, 6]
  Total extracted: 4168 (original dataset may differ due to skipped bad images)
All assertions passed.


In [12]:
# Cell 15 — Save features
torch.save(gnn_features,  "bracs_gnn_features-re.pt")
torch.save(labels_tensor, "bracs_labels-re.pt")   # authoritative label file

print(f"Saved → bracs_gnn_features-re.pt  {gnn_features.shape}")
print(f"Saved → bracs_labels-re.pt        {labels_tensor.shape}")
print(f"Class mapping: {raw_dataset.class_to_idx}")

Saved → bracs_gnn_features-re.pt  torch.Size([4168, 196, 1024])
Saved → bracs_labels-re.pt        torch.Size([4168])
Class mapping: {'0_N': 0, '1_PB': 1, '2_UDH': 2, '3_FEA': 3, '4_ADH': 4, '5_DCIS': 5, '6_IC': 6}
